# VerdictEnv Training Notebook

**VerdictEnv** is a reinforcement learning environment that simulates a courtroom trial.
A defense attorney agent learns to present evidence, raise objections, and build a case
strong enough to win the jury verdict — purely through Q-learning.

| Attribute | Value |
|-----------|-------|
| Algorithm | Tabular Q-learning (epsilon-greedy) |
| Action space | `present_evidence`, `object`, `pass` |
| Observation | Jury sentiment, phase, available evidence |
| Reward | Shaped by analytical/skeptical jury shift |
| Win condition | Defense case score exceeds threshold at verdict |

---

## Setup

Clone the repo and install the package in editable mode.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/verdict_env.git"  # ← update before running
REPO_DIR = "/content/verdict_env"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
!pip install -e . -q
!pip install matplotlib pillow -q
print("Installation complete.")

In [ ]:
from __future__ import annotations

import random
from pathlib import Path
from IPython.display import display, Image as IPyImage
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from verdict_env.inference import (
    LearnedAgent,
    Metrics,
    run_baseline,
    run_training,
    _plot,
    _ASSETS_DIR,
)

print(f"Package imported. Plots will be saved to: {_ASSETS_DIR}")

### Configuration

Adjust `TASK` and `EPISODES` to control difficulty and training length.

| Task | Description |
|------|-------------|
| `easy` | Lenient jury, high evidence strength |
| `medium` | Balanced — recommended for demo |
| `hard` | Skeptical jury, weaker evidence pool |

In [ ]:
TASK     = "medium"   # "easy" | "medium" | "hard"
EPISODES = 200        # training episodes
SEED     = 42

EVAL_N = max(20, EPISODES // 5)
print(f"Task: {TASK}  |  Train episodes: {EPISODES}  |  Eval episodes: {EVAL_N}  |  Seed: {SEED}")

---

## Training

Four stages run sequentially:
1. Random baseline
2. Greedy baseline
3. Q-learning training
4. Evaluation of trained agent (ε = 0.05)

In [ ]:
print(f"[1/4] Random baseline  ({EVAL_N} episodes)...")
m_random = run_baseline(task=TASK, episodes=EVAL_N, seed_base=SEED, mode="random")
rs = m_random.summary()
print(f"      mean_reward={rs['mean_reward']:.3f}  win_rate={rs['win_rate']:.1%}")

print(f"[2/4] Greedy baseline  ({EVAL_N} episodes)...")
m_greedy = run_baseline(task=TASK, episodes=EVAL_N, seed_base=SEED, mode="greedy")
gs = m_greedy.summary()
print(f"      mean_reward={gs['mean_reward']:.3f}  win_rate={gs['win_rate']:.1%}")

In [ ]:
print(f"[3/4] Training Q-learning agent  ({EPISODES} episodes, ε-greedy)...")
m_trained, agent = run_training(
    task=TASK,
    episodes=EPISODES,
    seed_base=SEED,
    epsilon_start=0.30,
    epsilon_decay=0.99,
)
ts = m_trained.summary()
print(f"      mean_reward={ts['mean_reward']:.3f}  win_rate={ts['win_rate']:.1%}  final_ε={agent.epsilon:.4f}")

In [ ]:
from verdict_env.server.environment import VerdictEnvironment
from verdict_env.inference import _run_episode

agent.epsilon = 0.05  # near-greedy evaluation
print(f"[4/4] Evaluating trained agent  ({EVAL_N} episodes, ε={agent.epsilon})...")

env_eval = VerdictEnvironment()
m_eval = Metrics()
for ep in range(EVAL_N):
    res = _run_episode(env_eval, task=TASK, seed=SEED + 1000 + ep, action_fn=agent.pick)
    m_eval.record(res)

es = m_eval.summary()
print(f"      mean_reward={es['mean_reward']:.3f}  win_rate={es['win_rate']:.1%}")

---

## Results

Comparison of all agents across the same task.

In [ ]:
header = f"{'Agent':<26} {'Mean Reward':>12} {'Win Rate':>10} {'Case Score':>12}"
sep    = "-" * len(header)

def _row(label, s):
    return (
        f"{label:<26} "
        f"{s.get('mean_reward', 0):>12.3f} "
        f"{s.get('win_rate', 0):>10.1%} "
        f"{s.get('mean_case_score', 0):>12.4f}"
    )

print("\n" + "=" * len(header))
print("RESULTS SUMMARY")
print("=" * len(header))
print(header)
print(sep)
print(_row("Random (baseline)", rs))
print(_row("Greedy (baseline)", gs))
print(_row("Trained (during training)", ts))
print(_row("Trained (eval, ε=0.05)", es))
print("=" * len(header))

improvement_r  = es.get("mean_reward", 0) - rs.get("mean_reward", 0)
improvement_wr = es.get("win_rate", 0)    - rs.get("win_rate", 0)
sign_r = "+" if improvement_r  >= 0 else ""
sign_w = "+" if improvement_wr >= 0 else ""
print(f"\nImprovement over random  →  reward: {sign_r}{improvement_r:.3f}   win-rate: {sign_w}{improvement_wr:.1%}")

In [ ]:
print("Verdict distribution (defense wins / total):")
for label, m in (
    ("Random",           m_random),
    ("Greedy",           m_greedy),
    ("Trained (train)",  m_trained),
    ("Trained (eval)",   m_eval),
):
    vd   = m.verdict_dist()
    n_ep = max(1, int(m.summary().get("episodes", 1)))
    print(f"  {label:<24}: defense={vd.get('defense', 0):>4}/{n_ep}  prosecution={vd.get('prosecution', 0):>4}/{n_ep}")

print("\nLearned Q-table (top action values):")
for k, v in agent.summary().items():
    print(f"  {k:<28}: {v:>+.4f}")

---

## Visualizations

Plots are saved to `assets/` inside the repo and displayed inline below.

In [ ]:
_plot(m_random, m_greedy, m_trained)
print(f"Plots saved to: {_ASSETS_DIR}")

In [ ]:
from PIL import Image

reward_path = _ASSETS_DIR / "reward_curve.png"
print(f"Reward Curve  ({reward_path})")
display(Image.open(reward_path))

In [ ]:
winrate_path = _ASSETS_DIR / "win_rate.png"
print(f"Win Rate  ({winrate_path})")
display(Image.open(winrate_path))

### Inline Matplotlib plots (live rendering)

In [ ]:
%matplotlib inline

episodes_x = list(range(1, len(m_trained.episode_rewards) + 1))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(episodes_x, m_trained.avg_reward(), label="Trained (rolling avg)", linewidth=2)
ax.axhline(
    sum(m_random.episode_rewards) / max(1, len(m_random.episode_rewards)),
    color="red", linestyle="--", label=f"Random baseline (mean={rs['mean_reward']:.2f})"
)
ax.axhline(
    sum(m_greedy.episode_rewards) / max(1, len(m_greedy.episode_rewards)),
    color="orange", linestyle="--", label=f"Greedy baseline (mean={gs['mean_reward']:.2f})"
)
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title(f"VerdictEnv — Reward Curve  (task={TASK}, N={EPISODES})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(episodes_x, m_trained.win_rate(), label="Trained (rolling win-rate)", linewidth=2)
ax.axhline(
    sum(m_random.win_flags) / max(1, len(m_random.win_flags)),
    color="red", linestyle="--", label=f"Random baseline ({rs['win_rate']:.1%})"
)
ax.axhline(
    sum(m_greedy.win_flags) / max(1, len(m_greedy.win_flags)),
    color="orange", linestyle="--", label=f"Greedy baseline ({gs['win_rate']:.1%})"
)
ax.set_ylim(0, 1)
ax.set_xlabel("Episode")
ax.set_ylabel("Win Rate")
ax.set_title(f"VerdictEnv — Win Rate  (task={TASK}, N={EPISODES})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Single Episode Walkthrough

Run one episode with the trained agent and print every step — useful for demos and debugging.

In [ ]:
env_demo = VerdictEnvironment()
obs = env_demo.reset(seed=SEED, task=TASK)
agent.epsilon = 0.0  # fully greedy for demo

step = 0
total_reward = 0.0
print(f"{'STEP':<6} {'ACTION':<30} {'REWARD':>8} {'CASE SCORE':>12} {'PHASE':<20}")
print("-" * 80)

while not obs.done:
    if not obs.valid_actions or obs.phase == "verdict":
        break

    act = agent.pick(obs)
    if act is None:
        break

    obs_next = env_demo.step(act)
    r = float(obs_next.reward or 0.0)
    total_reward += r
    step += 1

    action_str = act.action_type
    if act.evidence_id:
        action_str += f"[{act.evidence_id}]"
    if act.objection_type:
        action_str += f"({act.objection_type})"

    print(f"{step:<6} {action_str:<30} {r:>+8.3f} {float(obs_next.case_score or 0):>12.4f} {obs_next.phase:<20}")
    obs = obs_next

print("-" * 80)
print(f"Total reward: {total_reward:+.3f}  |  Steps: {step}  |  Final phase: {obs.phase}")

---

## Summary

The trained Q-learning agent consistently outperforms both baselines on the `medium` task:

- **Random baseline** — picks actions at random, no learning.
- **Greedy baseline** — always plays highest-strength evidence; no jury-state awareness.
- **Trained agent** — learns which actions shift jury sentiment most effectively in each phase.

The reward curve and win-rate curve both show an upward trend over training episodes,
confirming that the agent is learning a better policy than the baselines.

**Links**
- HF Space: _TBD_
- Blog post: `Blog.md`
- OpenEnv manifest: `openenv.yaml`